# 01 - HAM10000 Exploratory Data Analysis

Dataset overview, class balance, demographics, mask integrity and per-class samples for the HAM10000 segmentation dataset.

Training the Mask R-CNN model is handled by `scripts/train_mask_rcnn.py` (run from the repo root)—keeping the heavy compute out of Jupyter avoids kernel-crash issues over SSH.

In [20]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
from src.paths import HAM10000_DIR, EDA_HAM10000_DIR
from src.viz import setup_style, save_fig, HAM_CLASS_COLORS

In [ ]:
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

setup_style()
pd.set_option('display.max_columns', None)

## Exploratory Data Analysis

In [ ]:
# Load HAM10000 metadata
meta_df = pd.read_csv(os.path.join(HAM10000_DIR, "metadata.csv"))
n_images = len(os.listdir(os.path.join(HAM10000_DIR, "images")))
n_masks = len(os.listdir(os.path.join(HAM10000_DIR, "masks")))

print(f"Metadata rows: {len(meta_df):,}")
print(f"Images on disk: {n_images:,}")
print(f"Masks on disk:  {n_masks:,}")
print(f"\nColumns: {list(meta_df.columns)}")
meta_df.head()

#### 2.4.1 Class distribution (`dx`)

HAM10000 is famously imbalanced — `nv` (melanocytic nevi) dominates the dataset. Recognizing this early is important to decide whether to stratify the split, oversample, or weight the loss when training downstream classifiers.

In [ ]:
dx_counts = meta_df['dx'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    x=dx_counts.index, y=dx_counts.values,
    hue=dx_counts.index, palette=HAM_CLASS_COLORS,
    edgecolor='white', linewidth=1.5, legend=False, ax=ax,
)
for bar, val in zip(ax.patches, dx_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + dx_counts.max() * 0.012,
        f'{val:,}', ha='center', va='bottom', fontweight='bold',
    )
ax.set_xlabel('Diagnóstico (dx)')
ax.set_ylabel('Cantidad de imágenes')
ax.set_title('HAM10000 — Distribución de clases')
sns.despine(ax=ax)
plt.tight_layout()
save_fig(fig, EDA_HAM10000_DIR, '01_dx_distribution')
plt.show()

print("\nPorcentaje por clase:")
print((dx_counts / dx_counts.sum() * 100).round(2).to_string())

#### 2.4.2 Demographics: age, sex, localization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=meta_df, x='age', bins=20, kde=True, color='#2196F3', ax=axes[0])
axes[0].set_title('Edad')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Frecuencia')

sex_counts = meta_df['sex'].value_counts()
sns.barplot(
    x=sex_counts.index, y=sex_counts.values,
    hue=sex_counts.index,
    palette=['#5B9BD5', '#ED7D31', '#999999'][:len(sex_counts)],
    legend=False, ax=axes[1],
)
axes[1].set_title('Sexo')
axes[1].set_xlabel('')
axes[1].set_ylabel('Cantidad')

loc_counts = meta_df['localization'].value_counts()
sns.barplot(y=loc_counts.index, x=loc_counts.values, color='#4CAF50', ax=axes[2])
axes[2].set_title('Localización')
axes[2].set_xlabel('Cantidad')
axes[2].set_ylabel('')

fig.suptitle('HAM10000 — Demografía', y=1.02)
plt.tight_layout()
save_fig(fig, EDA_HAM10000_DIR, '02_demographics')
plt.show()

#### 2.4.3 Class × anatomical localization (heatmap %)

In [ ]:
heatmap_data = (
    meta_df.groupby(['dx', 'localization']).size()
    .reset_index(name='count')
    .pivot(index='dx', columns='localization', values='count')
    .fillna(0)
)
heatmap_pct = heatmap_data.div(heatmap_data.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    heatmap_pct, annot=True, fmt='.1f', cmap='Blues',
    linewidths=0.5, linecolor='#e0e0e0',
    cbar_kws={'label': '% dentro de la clase'}, ax=ax,
)
ax.set_title('HAM10000 — Localización por clase (%)')
ax.set_xlabel('Localización')
ax.set_ylabel('Diagnóstico')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
save_fig(fig, EDA_HAM10000_DIR, '03_dx_x_localization')
plt.show()

#### 2.4.4 Image dimensions and lesion area distribution

Sampled inspection (500 images) — checks dataset is uniform in size and gives a sense of how large lesions tend to be relative to the full image.

In [ ]:
random.seed(42)
sample_size = 500
sample_files = random.sample(os.listdir(os.path.join(HAM10000_DIR, "images")), sample_size)

dims, areas_px, areas_pct, empty_masks = [], [], [], 0
for fname in sample_files:
    img_id = os.path.splitext(fname)[0]
    img_path = os.path.join(HAM10000_DIR, "images", fname)
    mask_path = os.path.join(HAM10000_DIR, "masks", f"{img_id}_segmentation.png")
    with Image.open(img_path) as im:
        dims.append(im.size)
    if os.path.exists(mask_path):
        mask = np.array(Image.open(mask_path).convert('L'))
        binary = mask > 0
        n_px = int(binary.sum())
        if n_px == 0:
            empty_masks += 1
        areas_px.append(n_px)
        areas_pct.append(n_px / binary.size * 100)

dims_df = pd.DataFrame(dims, columns=['width', 'height'])
print(f"Sample size: {sample_size}")
print(f"Empty masks in sample: {empty_masks}")
print("\nImage dimensions:")
print(dims_df.describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(areas_px, bins=40, kde=True, color='#FF5722', ax=axes[0])
axes[0].set_title('Área de lesión (pixeles)')
axes[0].set_xlabel('Pixeles')
sns.histplot(areas_pct, bins=40, kde=True, color='#9C27B0', ax=axes[1])
axes[1].set_title('Área de lesión (% de imagen)')
axes[1].set_xlabel('% del total')
fig.suptitle(f'HAM10000 — Tamaño de lesión (muestra n={sample_size})', y=1.02)
plt.tight_layout()
save_fig(fig, EDA_HAM10000_DIR, '04_lesion_area')
plt.show()

#### 2.4.5 Sample images per class (image + ground-truth mask overlay)

In [ ]:
classes = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'df', 'vasc']
n_per_class = 2

fig, axes = plt.subplots(
    len(classes), n_per_class * 2,
    figsize=(4 * n_per_class * 2, 3 * len(classes)),
)
fig.suptitle('HAM10000 — Ejemplos por clase (imagen | overlay GT)', y=1.0)

for row, cls in enumerate(classes):
    cls_ids = (
        meta_df[meta_df['dx'] == cls]['image_id']
        .sample(n=n_per_class, random_state=42)
        .tolist()
    )
    for col_i, img_id in enumerate(cls_ids):
        img_path = os.path.join(HAM10000_DIR, "images", f"{img_id}.jpg")
        mask_path = os.path.join(HAM10000_DIR, "masks", f"{img_id}_segmentation.png")
        if not (os.path.exists(img_path) and os.path.exists(mask_path)):
            continue
        img_np = np.array(Image.open(img_path).convert('RGB'))
        mask_np = np.array(Image.open(mask_path).convert('L'))

        ax_img = axes[row, col_i * 2]
        ax_img.imshow(img_np)
        ax_img.set_title(f"{cls} — {img_id}", fontsize=9)
        ax_img.axis('off')

        ax_over = axes[row, col_i * 2 + 1]
        ax_over.imshow(img_np)
        ax_over.imshow(mask_np, alpha=0.4, cmap='Reds')
        ax_over.set_title('overlay GT', fontsize=9)
        ax_over.axis('off')

plt.tight_layout()
save_fig(fig, EDA_HAM10000_DIR, '05_samples_per_class')
plt.show()